<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/VLM-Experiments/Classifier%2BVLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch
!pip install -q timm
!pip install -q evaluate
!pip install -q peft
!pip install --upgrade -q torchao
!pip install -q scikit-learn
!pip install -q matplotlib seaborn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.7 MB/s eta 0:00:00


In [2]:
import os
import re
import gc
import time
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from datasets import load_dataset, DatasetDict

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from torchvision.transforms import (
    RandomResizedCrop,
    Resize,
    CenterCrop,
    Compose,
    Normalize,
    ToTensor,
)

from transformers import (
    DefaultDataCollator,
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoProcessor,
    AutoModelForImageTextToText,
    TrainingArguments,
    Trainer,
)

In [3]:
def get_image_size(image_processor):
    if "shortest_edge" in image_processor.size:
        return image_processor.size["shortest_edge"]
    return image_processor.size["height"]

def apply_transforms(examples, image_processor, is_train=True):
    image_size = get_image_size(image_processor)
    normalize = Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    )

    if is_train:
        transform = Compose([
            RandomResizedCrop(image_size),
            ToTensor(),
            normalize,
        ])
    else:
        transform = Compose([
            Resize(image_size),
            CenterCrop(image_size),
            ToTensor(),
            normalize,
        ])

    examples["pixel_values"] = [
        transform(img.convert("RGB")) for img in examples["image"]
    ]
    del examples["image"]
    return examples

In [4]:
# -----------------------------------------------------
# Shared metric helpers for ConvNeXt evaluation
# -----------------------------------------------------
# Trainer needs compute_metrics(eval_pred), while later analysis needs
# probability, confidence, and top-k helper functions.

def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def top_k_accuracy_from_probs(probs, labels, k=5):
    """Computes top-k accuracy from [N, C] class probabilities."""
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(int)
    k = min(k, probs.shape[1])
    top_k_predictions = np.argpartition(probs, -k, axis=1)[:, -k:]
    return float(np.mean(np.any(top_k_predictions == labels[:, None], axis=1)))


def compute_metrics(eval_pred):
    """Hugging Face Trainer-compatible metric function."""
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    labels_np = eval_pred.label_ids.astype(int)
    probs = softmax_np(logits)
    preds = np.argmax(probs, axis=1)

    return {
        "accuracy": accuracy_score(labels_np, preds),
        "macro_f1": f1_score(labels_np, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_np, preds, average="weighted", zero_division=0),
        "top_5_accuracy": top_k_accuracy_from_probs(probs, labels_np, k=5),
    }


def get_topk_info(probs_row, id2label_mapping, k):
    """
    Returns top-k ids, dataset labels, natural labels, and probabilities
    for one sample.
    """
    probs_row = np.asarray(probs_row)
    k = min(k, len(probs_row))
    top_ids = np.argsort(probs_row)[::-1][:k].astype(int).tolist()
    top_dataset_labels = [id2label_mapping[i] for i in top_ids]
    top_natural_labels = [id2label_mapping[i].replace("_", " ") for i in top_ids]
    top_probs = [float(probs_row[i]) for i in top_ids]

    return top_ids, top_dataset_labels, top_natural_labels, top_probs

print("Metric helpers defined.")

Metric helpers defined.


In [5]:
# -----------------------------
# Balanced Food101 subset config
# -----------------------------
DATASET_NAME = "ethz/food101"
NUM_SELECTED_CLASSES = 20
SAMPLES_PER_CLASS = 250
TEST_SIZE = 0.20
SEED = 42

# Load the full Food101 training split.
# We use the train split and create our own stratified train/validation split
# because this experiment is meant to compare fine-tuning strategies cheaply.
food_full_train = load_dataset(DATASET_NAME, split="train")
original_labels = food_full_train.features["label"].names

rng = random.Random(SEED)

# Group dataset indices by original Food101 class id
label_to_indices = defaultdict(list)
for idx, label_id in enumerate(food_full_train["label"]):
    label_to_indices[int(label_id)].append(idx)

# Keep only classes that have enough examples
eligible_label_ids = [
    label_id
    for label_id, indices in label_to_indices.items()
    if len(indices) >= SAMPLES_PER_CLASS
]

if len(eligible_label_ids) < NUM_SELECTED_CLASSES:
    raise ValueError(
        f"Only {len(eligible_label_ids)} classes have at least "
        f"{SAMPLES_PER_CLASS} samples. Need {NUM_SELECTED_CLASSES}."
    )

# Select 20 classes reproducibly
selected_old_label_ids = sorted(rng.sample(eligible_label_ids, NUM_SELECTED_CLASSES))

# Select exactly 250 images per selected class
selected_indices = []
for old_label_id in selected_old_label_ids:
    indices = label_to_indices[old_label_id].copy()
    rng.shuffle(indices)
    selected_indices.extend(indices[:SAMPLES_PER_CLASS])

rng.shuffle(selected_indices)

food_subset = food_full_train.select(selected_indices)

print(f"Total selected images: {len(food_subset)}")
print(f"Selected classes: {len(selected_old_label_ids)}")
print("Selected class names:")
for old_label_id in selected_old_label_ids:
    print(f"  old_id={old_label_id:3d} -> {original_labels[old_label_id]}")

# Stratified split while labels are still original Food101 label IDs
food = food_subset.train_test_split(
    test_size=TEST_SIZE,
    shuffle=True,
    seed=SEED,
    stratify_by_column="label",
)

# Remap selected original labels to contiguous labels 0..19.
# This is important because the classifier head will have exactly 20 outputs.
old_to_new = {
    old_label_id: new_label_id
    for new_label_id, old_label_id in enumerate(selected_old_label_ids)
}

new_to_old = {
    new_label_id: old_label_id
    for old_label_id, new_label_id in old_to_new.items()
}

labels = [
    original_labels[new_to_old[new_label_id]]
    for new_label_id in range(NUM_SELECTED_CLASSES)
]

id2label = {
    new_label_id: label_name
    for new_label_id, label_name in enumerate(labels)
}

label2id = {
    label_name: new_label_id
    for new_label_id, label_name in id2label.items()
}

def remap_label(example):
    example["label"] = old_to_new[int(example["label"])]
    return example

food = DatasetDict({
    "train": food["train"].map(remap_label),
    "test": food["test"].map(remap_label),
})

# Sanity checks
train_counts = Counter(food["train"]["label"])
test_counts = Counter(food["test"]["label"])

print("\nAfter remapping:")
print(f"Train size: {len(food['train'])}")
print(f"Validation size: {len(food['test'])}")
print(f"Number of labels: {len(labels)}")
print(f"Train class counts: {sorted(train_counts.items())}")
print(f"Validation class counts: {sorted(test_counts.items())}")

assert len(food["train"]) + len(food["test"]) == NUM_SELECTED_CLASSES * SAMPLES_PER_CLASS
assert set(train_counts.keys()) == set(range(NUM_SELECTED_CLASSES))
assert set(test_counts.keys()) == set(range(NUM_SELECTED_CLASSES))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

Total selected images: 5000
Selected classes: 20
Selected class names:
  old_id= 10 -> bruschetta
  old_id= 11 -> caesar_salad
  old_id= 13 -> caprese_salad
  old_id= 15 -> ceviche
  old_id= 21 -> chocolate_cake
  old_id= 24 -> clam_chowder
  old_id= 31 -> donuts
  old_id= 33 -> edamame
  old_id= 34 -> eggs_benedict
  old_id= 40 -> french_fries
  old_id= 44 -> fried_rice
  old_id= 53 -> hamburger
  old_id= 54 -> hot_and_sour_soup
  old_id= 58 -> ice_cream
  old_id= 59 -> lasagna
  old_id= 64 -> miso_soup
  old_id= 72 -> pancakes
  old_id= 74 -> peking_duck
  old_id= 97 -> takoyaki
  old_id= 99 -> tuna_tartare


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


After remapping:
Train size: 4000
Validation size: 1000
Number of labels: 20
Train class counts: [(0, 200), (1, 200), (2, 200), (3, 200), (4, 200), (5, 200), (6, 200), (7, 200), (8, 200), (9, 200), (10, 200), (11, 200), (12, 200), (13, 200), (14, 200), (15, 200), (16, 200), (17, 200), (18, 200), (19, 200)]
Validation class counts: [(0, 50), (1, 50), (2, 50), (3, 50), (4, 50), (5, 50), (6, 50), (7, 50), (8, 50), (9, 50), (10, 50), (11, 50), (12, 50), (13, 50), (14, 50), (15, 50), (16, 50), (17, 50), (18, 50), (19, 50)]


In [6]:
print(f"Number of selected labels: {len(labels)}")
print("id2label:")
for class_id, class_name in id2label.items():
    print(f"  {class_id}: {class_name}")

print("\nlabel2id:")
for class_name, class_id in label2id.items():
    print(f"  {class_name}: {class_id}")

Number of selected labels: 20
id2label:
  0: bruschetta
  1: caesar_salad
  2: caprese_salad
  3: ceviche
  4: chocolate_cake
  5: clam_chowder
  6: donuts
  7: edamame
  8: eggs_benedict
  9: french_fries
  10: fried_rice
  11: hamburger
  12: hot_and_sour_soup
  13: ice_cream
  14: lasagna
  15: miso_soup
  16: pancakes
  17: peking_duck
  18: takoyaki
  19: tuna_tartare

label2id:
  bruschetta: 0
  caesar_salad: 1
  caprese_salad: 2
  ceviche: 3
  chocolate_cake: 4
  clam_chowder: 5
  donuts: 6
  edamame: 7
  eggs_benedict: 8
  french_fries: 9
  fried_rice: 10
  hamburger: 11
  hot_and_sour_soup: 12
  ice_cream: 13
  lasagna: 14
  miso_soup: 15
  pancakes: 16
  peking_duck: 17
  takoyaki: 18
  tuna_tartare: 19


In [7]:
# ConvNeXt-Tiny model from Hugging Face
convnext_model_id_hf = "facebook/convnext-tiny-224"

# Load the ConvNeXt-specific image processor
convnext_image_processor = AutoImageProcessor.from_pretrained(convnext_model_id_hf)

# Apply ConvNeXt-specific transforms to the already split balanced Food101 subset
food_convnext = food.copy()

food_convnext["train"] = food_convnext["train"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=True
    )
)

food_convnext["test"] = food_convnext["test"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=False
    )
)

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

In [8]:
# Load the ConvNeXt-Tiny model with a new 20-class classification head
convnext_model_hf = AutoModelForImageClassification.from_pretrained(
    convnext_model_id_hf,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Ensure all parameters require gradients for full fine-tuning
for param in convnext_model_hf.parameters():
    param.requires_grad = True

print("ConvNeXt-Tiny model loaded and configured for full fine-tuning.")
print(convnext_model_hf)

# Sanity checks
print("Number of labels:", convnext_model_hf.config.num_labels)
print("id2label length:", len(convnext_model_hf.config.id2label))
print("label2id length:", len(convnext_model_hf.config.label2id))

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

[transformers] You passed `num_labels=20` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/114M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

[transformers] ConvNextForImageClassification LOAD REPORT from: facebook/convnext-tiny-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([20, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([20])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ConvNeXt-Tiny model loaded and configured for full fine-tuning.
ConvNextForImageClassification(
  (convnext): ConvNextModel(
    (embeddings): ConvNextEmbeddings(
      (patch_embeddings): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
    )
    (encoder): ConvNextEncoder(
      (stages): ModuleList(
        (0): ConvNextStage(
          (downsampling_layer): ModuleList()
          (layers): ModuleList(
            (0-2): 3 x ConvNextLayer(
              (dwconv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
              (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
              (pwconv1): Linear(in_features=96, out_features=384, bias=True)
              (act): GELUActivation()
              (pwconv2): Linear(in_features=384, out_features=96, bias=True)
              (drop_path): Identity()
            )
          )
        )
        (1): ConvN

In [ ]:
data_collator = DefaultDataCollator()

# Define TrainingArguments for ConvNeXt-Tiny full fine-tuning
training_args_convnext_hf = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_convnext_tiny_full_finetune",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_convnext_tiny_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

# Initialize the Trainer for ConvNeXt-Tiny
trainer_convnext_hf = Trainer(
    model=convnext_model_hf,
    args=training_args_convnext_hf,
    data_collator=data_collator,
    train_dataset=food_convnext["train"],
    eval_dataset=food_convnext["test"],
    processing_class=convnext_image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ConvNeXt-Tiny full model fine-tuning...")
train_results = trainer_convnext_hf.train()

Starting ConvNeXt-Tiny full model fine-tuning...


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Top 5 Accuracy
1,2.271125,2.062486,0.659000,0.647588,0.647588,0.926000
2,1.324854,1.124248,0.788000,0.783282,0.783282,0.952000
3,0.811847,0.769582,0.828000,0.827930,0.827930,0.966000
4,0.618937,0.623737,0.844000,0.844993,0.844993,0.971000
5,0.545841,0.554683,0.858000,0.858245,0.858245,0.972000
6,0.449548,0.486978,0.871000,0.871052,0.871052,0.981000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# -----------------------------------------------------
# ConvNeXt prediction analysis for VLM revalidation
# -----------------------------------------------------
# Goal:
#   1. Select ALL correct-but-low-confidence ConvNeXt samples.
#      For these, keep the ConvNeXt top-5 candidate classes.
#
#   2. Select ALL incorrect ConvNeXt samples.
#      For these, keep the ConvNeXt top-10 candidate classes.
#
# The VLM will later reclassify every selected sample using only that sample's
# candidate list.

print("Getting predictions from the trained ConvNeXt model...")
convnext_prediction_output = trainer_convnext_hf.predict(food_convnext["test"])

convnext_logits = convnext_prediction_output.predictions
if isinstance(convnext_logits, tuple):
    convnext_logits = convnext_logits[0]

true_labels = np.asarray(convnext_prediction_output.label_ids).astype(int)
convnext_probs = softmax_np(convnext_logits)
convnext_predictions = np.argmax(convnext_probs, axis=1).astype(int)

convnext_correct = convnext_predictions == true_labels
convnext_top1_confidence = np.max(convnext_probs, axis=1)

sorted_probs = np.sort(convnext_probs, axis=1)
convnext_top2_confidence = sorted_probs[:, -2] if convnext_probs.shape[1] > 1 else np.zeros_like(convnext_top1_confidence)
convnext_margin = convnext_top1_confidence - convnext_top2_confidence

convnext_top5_accuracy = top_k_accuracy_from_probs(convnext_probs, true_labels, k=5)
convnext_accuracy = float(np.mean(convnext_correct))

print(f"ConvNeXt accuracy:       {convnext_accuracy:.4f}")
print(f"ConvNeXt top-5 accuracy: {convnext_top5_accuracy:.4f}")

# -----------------------------
# Selection settings
# -----------------------------
# This threshold defines "low confidence but correct".
# Adjust it if you want a larger/smaller borderline set.
LOW_CONFIDENCE_CORRECT_THRESHOLD = 0.60

low_conf_correct_indices = np.where(
    convnext_correct & (convnext_top1_confidence < LOW_CONFIDENCE_CORRECT_THRESHOLD)
)[0]

incorrect_indices = np.where(~convnext_correct)[0]

print(f"\nLow-confidence correct threshold: {LOW_CONFIDENCE_CORRECT_THRESHOLD:.2f}")
print(f"Number of low-confidence correct samples selected: {len(low_conf_correct_indices)}")
print(f"Number of incorrect samples selected:              {len(incorrect_indices)}")
print(f"Total samples selected for VLM revalidation:       {len(low_conf_correct_indices) + len(incorrect_indices)}")

# -----------------------------
# Build selected_cases_df
# -----------------------------
# Correct low-confidence cases get top-5 candidates.
# Incorrect cases get top-10 candidates.
selected_rows = []

for sample_index in low_conf_correct_indices:
    sample_index = int(sample_index)
    top_ids, top_dataset_labels, top_natural_labels, top_probs = get_topk_info(
        convnext_probs[sample_index],
        id2label,
        k=5,
    )

    selected_rows.append({
        "sample_index": sample_index,
        "case_type": "low_confidence_correct",
        "candidate_k": 5,
        "true_label_id": int(true_labels[sample_index]),
        "true_label": id2label[int(true_labels[sample_index])],
        "convnext_pred_id": int(convnext_predictions[sample_index]),
        "convnext_pred_label": id2label[int(convnext_predictions[sample_index])],
        "convnext_correct": bool(convnext_correct[sample_index]),
        "convnext_top1_confidence": float(convnext_top1_confidence[sample_index]),
        "convnext_top2_confidence": float(convnext_top2_confidence[sample_index]),
        "convnext_margin": float(convnext_margin[sample_index]),
        "candidate_label_ids": top_ids,
        "candidate_dataset_labels": top_dataset_labels,
        "candidate_natural_labels": top_natural_labels,
        "candidate_confidences": top_probs,
        "true_label_in_candidates": int(true_labels[sample_index]) in top_ids,
    })

for sample_index in incorrect_indices:
    sample_index = int(sample_index)
    top_ids, top_dataset_labels, top_natural_labels, top_probs = get_topk_info(
        convnext_probs[sample_index],
        id2label,
        k=10,
    )

    selected_rows.append({
        "sample_index": sample_index,
        "case_type": "incorrect",
        "candidate_k": 10,
        "true_label_id": int(true_labels[sample_index]),
        "true_label": id2label[int(true_labels[sample_index])],
        "convnext_pred_id": int(convnext_predictions[sample_index]),
        "convnext_pred_label": id2label[int(convnext_predictions[sample_index])],
        "convnext_correct": bool(convnext_correct[sample_index]),
        "convnext_top1_confidence": float(convnext_top1_confidence[sample_index]),
        "convnext_top2_confidence": float(convnext_top2_confidence[sample_index]),
        "convnext_margin": float(convnext_margin[sample_index]),
        "candidate_label_ids": top_ids,
        "candidate_dataset_labels": top_dataset_labels,
        "candidate_natural_labels": top_natural_labels,
        "candidate_confidences": top_probs,
        "true_label_in_candidates": int(true_labels[sample_index]) in top_ids,
    })

selected_cases_df = pd.DataFrame(selected_rows)

print("\nSelected case counts:")
display(selected_cases_df["case_type"].value_counts().to_frame("count"))

print("\nTrue label coverage inside candidate lists:")
display(
    selected_cases_df
    .groupby("case_type")["true_label_in_candidates"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "coverage_rate", "sum": "covered", "count": "total"})
    .round(4)
)

print("\nPreview of selected cases for VLM revalidation:")
display(
    selected_cases_df[[
        "sample_index",
        "case_type",
        "candidate_k",
        "true_label",
        "convnext_pred_label",
        "convnext_top1_confidence",
        "convnext_margin",
        "true_label_in_candidates",
        "candidate_dataset_labels",
        "candidate_confidences",
    ]].head(20)
)

In [ ]:
# SmolVLM model from Hugging Face.
# This is used as a zero-shot VLM classifier by scoring each candidate class label
# with image-conditioned log-likelihood, not by generating free-form text.
smolvlm_model_id = "HuggingFaceTB/SmolVLM-Instruct"

vlm_device = "cuda" if torch.cuda.is_available() else "cpu"
vlm_dtype = torch.float16 if vlm_device == "cuda" else torch.float32

processor_smolvlm = AutoProcessor.from_pretrained(smolvlm_model_id)
model_smolvlm = AutoModelForImageTextToText.from_pretrained(
    smolvlm_model_id,
    torch_dtype=vlm_dtype,
)
model_smolvlm.to(vlm_device)
model_smolvlm.eval()

print(f"SmolVLM processor and model loaded on {vlm_device} with dtype={vlm_dtype}.")

In [ ]:
# -----------------------------------------------------
# VLM candidate-list revalidation helpers
# -----------------------------------------------------
# The VLM is NOT asked to classify from all 20 labels here.
# For each selected ConvNeXt case:
#   - low-confidence correct samples get the ConvNeXt top-5 candidates
#   - incorrect samples get the ConvNeXt top-10 candidates
# The VLM must choose exactly one label from that candidate list.


def vlm_label_name(label_name):
    return str(label_name).replace("_", " ")


def normalize_label_text(text):
    text = str(text).lower().strip()
    text = text.replace("_", " ")
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    prefixes = [
        "the answer is",
        "answer is",
        "answer",
        "the class is",
        "class is",
        "class",
        "the label is",
        "label is",
        "label",
        "the food is",
        "food is",
        "food",
        "this is",
        "it is",
        "it looks like",
    ]

    for prefix in prefixes:
        if text.startswith(prefix + " "):
            text = text[len(prefix):].strip()

    return text


def build_candidate_label_map(candidate_label_ids, id2label_mapping):
    """
    Builds a parser map restricted to the current candidate list.
    This prevents the VLM from being credited for labels outside the allowed list.
    """
    candidate_label_ids = [int(x) for x in candidate_label_ids]

    id_to_dataset_label = {
        label_id: id2label_mapping[label_id]
        for label_id in candidate_label_ids
    }

    id_to_natural_label = {
        label_id: vlm_label_name(id2label_mapping[label_id])
        for label_id in candidate_label_ids
    }

    normalized_to_id = {
        normalize_label_text(natural_label): label_id
        for label_id, natural_label in id_to_natural_label.items()
    }

    return {
        "candidate_label_ids": candidate_label_ids,
        "id_to_dataset_label": id_to_dataset_label,
        "id_to_natural_label": id_to_natural_label,
        "normalized_to_id": normalized_to_id,
        "candidate_natural_labels": [id_to_natural_label[i] for i in candidate_label_ids],
        "candidate_dataset_labels": [id_to_dataset_label[i] for i in candidate_label_ids],
    }


def parse_vlm_output_against_candidates(generated_text, candidate_label_ids, id2label_mapping):
    """
    Parses generated text into a class ID, but only if the generated label is
    one of the allowed candidate labels for that sample.
    """
    label_map = build_candidate_label_map(candidate_label_ids, id2label_mapping)
    normalized_output = normalize_label_text(generated_text)
    normalized_to_id = label_map["normalized_to_id"]

    # Exact match first.
    if normalized_output in normalized_to_id:
        pred_id = normalized_to_id[normalized_output]
        return pred_id, id2label_mapping[pred_id], normalized_output, True

    # Substring match, longest candidate first.
    candidates = sorted(
        normalized_to_id.items(),
        key=lambda x: len(x[0]),
        reverse=True,
    )

    matches = []
    for normalized_label, label_id in candidates:
        pos = normalized_output.find(normalized_label)
        if pos != -1:
            matches.append((pos, -len(normalized_label), label_id))

    if matches:
        matches.sort()
        _, _, pred_id = matches[0]
        return pred_id, id2label_mapping[pred_id], normalized_output, True

    return -1, "INVALID_OUTPUT", normalized_output, False


def build_vlm_candidate_rerank_prompt(candidate_label_ids, id2label_mapping, processor=None):
    """
    Builds a prompt where the VLM must choose from only this sample's
    ConvNeXt candidate labels.
    """
    label_map = build_candidate_label_map(candidate_label_ids, id2label_mapping)
    candidate_lines = "\n".join([
        f"- {name}"
        for name in label_map["candidate_natural_labels"]
    ])

    user_text = (
        "You are a food image classifier.\n"
        "Choose the single best class for the image from this candidate list:\n"
        f"{candidate_lines}\n\n"
        "Answer with exactly one class name from the candidate list.\n"
        "Do not explain.\n"
        "Answer:"
    )

    if processor is not None and hasattr(processor, "apply_chat_template"):
        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": user_text},
                    ],
                }
            ]
            return processor.apply_chat_template(messages, add_generation_prompt=True)
        except Exception as e:
            print("Chat template failed; falling back to manual <image> prompt.")
            print("Reason:", e)

    return "<image>\n" + user_text


def move_processor_inputs_to_device(inputs, device, dtype=None):
    moved = {}
    for key, value in inputs.items():
        value = value.to(device)
        if dtype is not None and torch.is_floating_point(value):
            value = value.to(dtype=dtype)
        moved[key] = value
    return moved


def generate_vlm_candidate_choice(
    model,
    processor,
    image,
    candidate_label_ids,
    id2label_mapping,
    device,
    dtype=None,
    max_new_tokens=12,
):
    """
    Runs one VLM generation call for one selected sample.
    The output is parsed only against the sample's allowed candidate labels.
    """
    image = image.convert("RGB")
    prompt = build_vlm_candidate_rerank_prompt(
        candidate_label_ids=candidate_label_ids,
        id2label_mapping=id2label_mapping,
        processor=processor,
    )

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    )
    inputs = move_processor_inputs_to_device(inputs, device=device, dtype=dtype)

    input_ids = inputs.get("input_ids", None)
    input_length = input_ids.shape[-1] if input_ids is not None else None

    tokenizer = getattr(processor, "tokenizer", None)
    eos_token_id = getattr(tokenizer, "eos_token_id", None) if tokenizer is not None else None
    pad_token_id = getattr(tokenizer, "pad_token_id", None) if tokenizer is not None else None
    if pad_token_id is None:
        pad_token_id = eos_token_id

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )

    if input_length is not None and generated_ids.shape[-1] > input_length:
        new_token_ids = generated_ids[:, input_length:]
    else:
        new_token_ids = generated_ids

    raw_output = processor.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0].strip()

    pred_id, pred_label, normalized_output, valid_output = parse_vlm_output_against_candidates(
        generated_text=raw_output,
        candidate_label_ids=candidate_label_ids,
        id2label_mapping=id2label_mapping,
    )

    return {
        "vlm_pred_id": int(pred_id),
        "vlm_pred_label": pred_label,
        "vlm_valid_output": bool(valid_output),
        "vlm_raw_output": raw_output,
        "vlm_normalized_output": normalized_output,
    }

print("VLM candidate-list revalidation helpers defined.")

In [ ]:
# -----------------------------------------------------
# Run VLM revalidation on ALL selected ConvNeXt cases
# -----------------------------------------------------
# This cell revalidates:
#   1. ALL low-confidence correct ConvNeXt samples using top-5 candidates.
#   2. ALL incorrect ConvNeXt samples using top-10 candidates.
#
# It then measures:
#   - whether VLM preserves the low-confidence correct predictions
#   - whether VLM fixes the incorrect ConvNeXt predictions
#   - whether VLM regresses previously correct predictions
#   - whether the true label was even available in the candidate list

assert "selected_cases_df" in globals(), "Run the ConvNeXt selection-analysis cell first."

print(f"Running VLM revalidation on {len(selected_cases_df)} selected cases...")
print(selected_cases_df["case_type"].value_counts())

model_smolvlm.eval()

rerank_rows = []
start_time = time.time()

for _, row in tqdm(selected_cases_df.iterrows(), total=len(selected_cases_df), desc="VLM revalidating selected cases"):
    sample_index = int(row["sample_index"])
    item = food["test"][sample_index]  # raw image dataset, not transformed ConvNeXt dataset

    candidate_label_ids = [int(x) for x in row["candidate_label_ids"]]

    vlm_result = generate_vlm_candidate_choice(
        model=model_smolvlm,
        processor=processor_smolvlm,
        image=item["image"],
        candidate_label_ids=candidate_label_ids,
        id2label_mapping=id2label,
        device=vlm_device,
        dtype=vlm_dtype,
        max_new_tokens=12,
    )

    true_id = int(row["true_label_id"])
    convnext_pred_id = int(row["convnext_pred_id"])
    vlm_pred_id = int(vlm_result["vlm_pred_id"])

    vlm_correct = vlm_pred_id == true_id
    convnext_correct_this_case = bool(row["convnext_correct"])

    # For low-confidence correct cases, a VLM mistake is a regression.
    vlm_preserved_correct = convnext_correct_this_case and vlm_correct
    vlm_regressed_correct = convnext_correct_this_case and (not vlm_correct)

    # For incorrect cases, a VLM correct answer is a fix.
    vlm_fixed_convnext_error = (not convnext_correct_this_case) and vlm_correct
    vlm_kept_same_wrong_prediction = (
        (not convnext_correct_this_case)
        and vlm_result["vlm_valid_output"]
        and (vlm_pred_id == convnext_pred_id)
    )
    vlm_changed_to_different_wrong_prediction = (
        (not convnext_correct_this_case)
        and vlm_result["vlm_valid_output"]
        and (vlm_pred_id != convnext_pred_id)
        and (vlm_pred_id != true_id)
    )

    rerank_rows.append({
        **row.to_dict(),
        **vlm_result,
        "vlm_correct": bool(vlm_correct),
        "vlm_preserved_correct": bool(vlm_preserved_correct),
        "vlm_regressed_correct": bool(vlm_regressed_correct),
        "vlm_fixed_convnext_error": bool(vlm_fixed_convnext_error),
        "vlm_kept_same_wrong_prediction": bool(vlm_kept_same_wrong_prediction),
        "vlm_changed_to_different_wrong_prediction": bool(vlm_changed_to_different_wrong_prediction),
    })

vlm_revalidation_time = time.time() - start_time
vlm_revalidation_df = pd.DataFrame(rerank_rows)

print(f"\nVLM revalidation finished in {vlm_revalidation_time:.2f} seconds.")
print(f"Average latency per selected sample: {vlm_revalidation_time / max(len(vlm_revalidation_df), 1):.2f} seconds")

# -----------------------------
# Summary
# -----------------------------
low_conf_df = vlm_revalidation_df[vlm_revalidation_df["case_type"] == "low_confidence_correct"]
incorrect_df = vlm_revalidation_df[vlm_revalidation_df["case_type"] == "incorrect"]

summary = {
    "total_selected_cases": len(vlm_revalidation_df),
    "low_confidence_correct_cases": len(low_conf_df),
    "incorrect_cases": len(incorrect_df),
    "vlm_invalid_outputs": int((~vlm_revalidation_df["vlm_valid_output"]).sum()),
    "low_conf_correct_preserved_by_vlm": int(low_conf_df["vlm_preserved_correct"].sum()) if len(low_conf_df) else 0,
    "low_conf_correct_regressed_by_vlm": int(low_conf_df["vlm_regressed_correct"].sum()) if len(low_conf_df) else 0,
    "incorrect_cases_fixed_by_vlm": int(incorrect_df["vlm_fixed_convnext_error"].sum()) if len(incorrect_df) else 0,
    "incorrect_cases_same_wrong_by_vlm": int(incorrect_df["vlm_kept_same_wrong_prediction"].sum()) if len(incorrect_df) else 0,
    "incorrect_cases_different_wrong_by_vlm": int(incorrect_df["vlm_changed_to_different_wrong_prediction"].sum()) if len(incorrect_df) else 0,
    "incorrect_true_label_candidate_coverage": float(incorrect_df["true_label_in_candidates"].mean()) if len(incorrect_df) else np.nan,
    "net_change_on_selected_cases": (
        int(incorrect_df["vlm_fixed_convnext_error"].sum()) if len(incorrect_df) else 0
    ) - (
        int(low_conf_df["vlm_regressed_correct"].sum()) if len(low_conf_df) else 0
    ),
}

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
print("\n--- VLM Revalidation Summary ---")
display(summary_df)

print("\n--- Breakdown by case type ---")
display(
    vlm_revalidation_df
    .groupby("case_type")[[
        "true_label_in_candidates",
        "vlm_valid_output",
        "vlm_correct",
    ]]
    .mean()
    .round(4)
)

print("\n--- VLM fixed ConvNeXt incorrect predictions ---")
fixed_df = vlm_revalidation_df[vlm_revalidation_df["vlm_fixed_convnext_error"] == True]
if len(fixed_df) > 0:
    display(fixed_df[[
        "sample_index",
        "true_label",
        "convnext_pred_label",
        "vlm_pred_label",
        "convnext_top1_confidence",
        "convnext_margin",
        "candidate_dataset_labels",
        "candidate_confidences",
        "vlm_raw_output",
    ]].head(30))
else:
    print("No ConvNeXt mistakes were fixed by the VLM.")

print("\n--- VLM regressed low-confidence correct ConvNeXt predictions ---")
regressed_df = vlm_revalidation_df[vlm_revalidation_df["vlm_regressed_correct"] == True]
if len(regressed_df) > 0:
    display(regressed_df[[
        "sample_index",
        "true_label",
        "convnext_pred_label",
        "vlm_pred_label",
        "convnext_top1_confidence",
        "convnext_margin",
        "candidate_dataset_labels",
        "candidate_confidences",
        "vlm_raw_output",
    ]].head(30))
else:
    print("No low-confidence correct ConvNeXt predictions were regressed by the VLM.")

print("\n--- Preview of all VLM revalidation results ---")
display(vlm_revalidation_df[[
    "sample_index",
    "case_type",
    "candidate_k",
    "true_label",
    "convnext_pred_label",
    "vlm_pred_label",
    "convnext_top1_confidence",
    "convnext_margin",
    "true_label_in_candidates",
    "vlm_valid_output",
    "vlm_correct",
    "vlm_fixed_convnext_error",
    "vlm_regressed_correct",
    "candidate_dataset_labels",
    "candidate_confidences",
    "vlm_raw_output",
]].head(50))